In [ ]:
import pandas as pd
from pathlib import Path

# ================================================
# CONFIG
# ================================================
FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")
MASTER_SHEET = "Master Data "
CATEGORY_SHEET = "Sheet1"               # the sheet that has PARTNO, Category

MAX_HOURS_PER_DAY    = 22.0
CHANGEOVER_HOURS     = 40 / 60.0
MAX_PARTS_PER_MACHINE = 3
TARGET_COVERAGE_DAYS = 3.0

# Known machine patterns (update after seeing real data)
KNOWN_MACHINES = {
    "MP-01", "MP-04", "MP-05", "MP-08", "MP-10", "MP-11", "MP-17",
    "TOYO-IST", "TOYO1ST", "TOYO IST", "VT-120T", "JSW-55T", "FANUC-50T"
}

# ================================================
# MACHINE NAME NORMALIZATION
# ================================================
def normalize_machine(s):
    if pd.isna(s) or not str(s).strip():
        return None
    s = str(s).strip().upper()
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace("TOYOI", "TOYO").replace("TOYO ", "TOYO-").replace("TOYOIST", "TOYO-IST")
    s = s.replace(" ", "-")
    return s

def get_machines(cell):
    if pd.isna(cell):
        return []
    parts = str(cell).split(",")
    cleaned = [normalize_machine(x) for x in parts if normalize_machine(x)]
    return [m for m in set(cleaned) if m in KNOWN_MACHINES or "MP-" in m or "TOYO" in m]

# ================================================
# 1. Load category mapping (PARTNO → Category)
# ================================================
print("Loading category mapping from Sheet1...")
cat_df = pd.read_excel(FILE_PATH, sheet_name=CATEGORY_SHEET)
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_df = cat_df.rename(columns={"PARTNO": "Child Part"})
cat_map = dict(zip(cat_df["Child Part"], cat_df["Category"]))

print(f"Loaded {len(cat_map):,} part → category mappings")
print("Example categories:", list(cat_map.items())[:8])

# ================================================
# 2. Load & aggregate Master Data
# ================================================
print("\nLoading Master Data sheet...")
master = pd.read_excel(FILE_PATH, sheet_name=MASTER_SHEET)

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

print("Aggregating...")
agg_records = []

for child, g in master.groupby("Child Part"):
    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    min_qty = g["Minimum Quantity"].iloc[0]
    inv = g["Inventory_25"].iloc[0]
    net_req = daily_demand + min_qty - inv
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    cycle_sec = cycle_valid.iloc[0] if not cycle_valid.empty else 0
    if cycle_sec <= 0:
        continue

    vm_raw = g["Vertical Machines"].dropna().unique()
    vm_str = ",".join(vm_raw.astype(str))
    machines = get_machines(vm_str)
    if not machines:
        continue

    agg_records.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Raw_Category": cat_map.get(child, "Unknown")
    })

df = pd.DataFrame(agg_records)
print(f"\nValid parts after filtering: {len(df):,}")

# Use the real category from Sheet1
df["Category"] = df["Raw_Category"]
print("\nFinal categories used (from Sheet1):")
print(df["Category"].value_counts(dropna=False))

# Only plan Repeater + Stranger
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()
print(f"\nParts to schedule: {len(to_schedule):,}")

if len(to_schedule) == 0:
    print("No Repeater or Stranger parts found. Check Sheet1 categories or filtering.")
    exit()

# ================================================
# 3. Scheduler – simple greedy (big lots first)
# ================================================
machine_load = {m: 0.0 for m in KNOWN_MACHINES}
machine_sequence = {m: [] for m in KNOWN_MACHINES}
schedule = []

print("\nScheduling Repeaters & Strangers (largest net required first)...")
to_schedule = to_schedule.sort_values("Net_Required", ascending=False)

for _, row in to_schedule.iterrows():
    hrs_per_pc = row["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_left = row["Net_Required"]
    eligible = row["Eligible_Machines"]

    for m in sorted(eligible, key=lambda x: machine_load.get(x, 0)):
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE_DAY:
            continue

        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_left, max_qty)

        if assign_qty < 10:  # minimal meaningful lot
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        schedule.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# ================================================
# 4. Output
# ================================================
print("\n" + "="*100)
print("FINAL PRODUCTION PLAN – Only Repeaters & Strangers")
print("="*100)

total_hours = sum(machine_load.values())
total_parts = len(schedule)

for m in sorted(machine_load):
    seq = machine_sequence.get(m, [])
    h = machine_load.get(m, 0.0)
    if h < 0.1 and not seq:
        continue

    print(f"\n🛠 {m:10}   {h:6.1f} / {MAX_HOURS_PER_DAY:.1f} h   ({h/MAX_HOURS_PER_DAY*100:5.1f}%)")
    for p in seq:
        print(f"   • {p['Child Part']:22}   {p['Qty']:>7,} pcs   {p['Hours']:>5.1f}h   {p['Category']}")

print("\n" + "-"*100)
print(f"Total hours used : {total_hours:.1f} h")
print(f"Parts scheduled  : {total_parts}")
print("-"*100)

if schedule:
    pd.DataFrame(schedule).to_excel("production_plan_repeaters_strangers.xlsx", index=False)
    print("Saved → production_plan_repeaters_strangers.xlsx")
else:
    print("No parts scheduled – likely machine name mismatch or no eligible machines")